# Density Split Statistics Examples

This notebook demonstrates how to use the Density Split Statistics estimator from the ACM package.

Density split statistics divide the galaxy sample into quantiles based on the local density field and measure clustering separately for each quantile, providing information about non-Gaussian features.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

%matplotlib inline
%config InlineBackend.figure_format = "retina"

from helpers import load_estimator_parameters, make_lagrangian_mock

from acm import setup_logging
from acm.estimators.galaxy_clustering.backends.jaxpower import (
    JaxpowerBackend,  # noqa: F401 - register backend
)
from acm.estimators.galaxy_clustering.density_split import DensitySplit

setup_logging()

## Setting up the Density Split Estimator

We first create the density field and split it into quantiles based on local density values.

In [ ]:
los = "z"
data_positions, boxsize = make_lagrangian_mock(boxsize=500.0, los=los)

estimator = DensitySplit(
    backend="jaxpower",
    data_positions=data_positions,
    boxsize=boxsize, # Backend initialization arguments
    cellsize=5.0,
)
# Set the density contrast field with a smoothing radius
estimator.backend.set_density_contrast(smoothing_radius=10.0) # Mpc/h
# Split into quantiles
estimator.set_quantiles(nquantiles=5, method="randoms")

print(f"Divided sample into {estimator.nquantiles} density quantiles")

# Call the helper function to visualize the density split quantile split
estimator.plot_quantiles(
    nquantiles=estimator.nquantiles,
    delta_query=estimator._density_contrast_query,
    quantiles_idx=estimator._quantiles_idx
)

*NOTE: `_density_contrast_query` and `_quantiles_idx` are private attributes, that only exist to make the plotting easier. They are not part of the public API.*

## Computing Density Split statistics

To compute the different statistics, we call the `compute` method, with at least two information that determine the type of statistic to compute:
- `data_type`: either `'correlation'` or `'power'`, determines if the statistic to compute is a correlation function or a power spetrum (respectively)
- `cross`: either `True` or `False`, determines if the statistic to compute is the auto (quantile-quantile) or cross (quantile-data) -statistic

### Auto-Correlation Functions by Quantile

Compute the two-point correlation function for each density quantile.

In [ ]:
# args = load_estimator_parameters("ds_xiqq")["compute"]
# print(f"Computing acf with parameters: {args}") # cross=False, data_type="correlation"

result = estimator.compute(
    data_type="correlation",
    cross=False,
    edges=[np.arange(0, 151, 1), np.linspace(-1, 1, 120)],
    los=los,
    mode="smu",
    compute_sepsavg=False,
)

# plot the results
estimator.plot(result)

## Extracting Multipoles

We can extract specific multipoles (monopole, quadrupole) for each quantile.
The object returned is a `lsstypes.ObservableTree`:

In [ ]:
# Extract multipoles for each quantile
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

s = result.flatten(level=None)[0].coords("s") # s is the same for all quantiles
for i in range(estimator.nquantiles):
    quantile = result.get(quantiles=i)
    multipoles = quantile.project(ells=(0, 2))
    # Plot monopole
    axes[0].plot(s, s**2 * multipoles.get(ells=0), label=f'Q{i+1}', alpha=0.8)
    # Plot quadrupole
    axes[1].plot(s, s**2 * multipoles.get(ells=2), label=f'Q{i+1}', alpha=0.8)

# Add labels and legends
for i, ax in enumerate(axes):
    ax.set_xlabel(r"$s$ [Mpc/h]")
    ax.set_ylabel(r"$s^2 \xi_\ell(s)$")
    ax.set_title("Monopole" if i == 0 else "Quadrupole")
    ax.grid(True, alpha=0.3)
    ax.legend()

## Cross-Correlation with Full Sample

Compute the cross-correlation between each quantile and the full galaxy sample.

In [ ]:
args = load_estimator_parameters("ds_xiqg")["compute"]
print(f"Computing acf with parameters: {args}") # cross=True, data_type="correlation"

result = estimator.compute(los=los, **args)

estimator.plot(result, ell=0)

## Comparing Different Backends

The DensitySplit estimator supports both `jaxpower` and `pypower` backends for the correlations, only `jaxpower` for the power spectra.

In [ ]:
# Compute with both backends
from acm.estimators.galaxy_clustering.backends.pypower import (
    PypowerBackend,  # noqa: F401 - register backend
)

backends = ['jaxpower', 'pypower']
acf_dict = {}

fig, ax = plt.subplots(figsize=(8, 4))

args = load_estimator_parameters("ds_xiqq")["compute"]
for backend in backends:
    ds = DensitySplit(
        backend=backend,
        data_positions=data_positions,
        boxsize=boxsize,
        cellsize=5.0,
    )
    ds.backend.set_density_contrast(smoothing_radius=10.0) # Mpc/h
    ds.set_quantiles(nquantiles=5, method='randoms')
    result = ds.compute(los=los, **args)
    acf_dict[backend] = result

    # Compare monopole for the lowest density quantile (Q1)
    ls = '-' if backend == 'jaxpower' else '--'
    pole = result.get(quantiles=0).project(ells=0)
    s = pole.coords("s")
    ax.plot(s, s**2 * pole, label=backend, ls=ls, linewidth=2, alpha=0.8)

ax.set_xlabel('s [Mpc/h]')
ax.set_ylabel(r'$s^2 \xi_0(s)$ [Mpc/h]$^2$')
ax.set_title('Backend Comparison (Q1 Monopole)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Effect of Smoothing Scale

The smoothing radius affects how the density field is defined. Let's compare different smoothing scales.

In [ ]:
# Test different smoothing radii
smoothing_radii = [5, 10, 20]  # Mpc/h
colors = ['blue', 'green', 'red']

fig, ax = plt.subplots(figsize=(8, 5))

ds = DensitySplit(
    backend='jaxpower',
    data_positions=data_positions,
    boxsize=boxsize,
    cellsize=5.0,
)
args = load_estimator_parameters("ds_xiqq")["compute"]
for R, color in zip(smoothing_radii, colors, strict=True):
    ds.backend.set_density_contrast(smoothing_radius=R)
    ds.set_quantiles(nquantiles=3, method='randoms')
    result = ds.compute(los=los, **args)

    # Plot monopole for middle quantile
    pole = result.get(quantiles=1).project(ells=0)
    s = pole.coords("s")
    ax.plot(s, s**2 * pole, label=f'R = {R} Mpc/h', color=color, linewidth=2, alpha=0.8)

ax.set_xlabel('s [Mpc/h]')
ax.set_ylabel(r'$s^2 \xi_0(s)$ [Mpc/h]$^2$')
ax.set_title('Smoothing Scale Comparison (Middle Quantile)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()